In [0]:
# ============================================================================
# 1. IMPORTACIÓN DE LIBRERÍAS
# ============================================================================
from pyspark.sql.functions import col, current_timestamp, input_file_name, to_date, sum as _sum, countDistinct, approx_count_distinct, floor, datediff, lit, when
from delta.tables import DeltaTable
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType,TimestampType, DateType, DecimalType

# ============================================================================
# 2. CONFIGURACIÓN DE RUTAS Y PARÁMETROS
# ============================================================================
CATALOG = "workspace"
BRONCE_SCHEMA = "footloose_bronce"
SILVER_SCHEMA = "footloose_silver"
GOLD_SCHEMA = "footloose_gold"

# Fecha de corte para calcular la edad (fin del período analizado).
FECHA_CORTE = "2025-12-31"

raw_path =  f"/Volumes/{CATALOG}/{BRONCE_SCHEMA}/raw_data/"

CHECKPOINTS = f"/Volumes/{CATALOG}/{BRONCE_SCHEMA}/checkpoints/"
SCHEMAS = f"{CHECKPOINTS}/schemas"


checkpoint_path = f"{CHECKPOINTS}"

# Esquemas para los DataFrames

schema_transacciones = StructType([
    StructField("id_transaccion", StringType()),
    StructField("id_cliente", StringType()),
    StructField("id_producto", StringType()),
    StructField("estado_venta", StringType()),
    StructField("metodo_pago", StringType()),
    StructField("canal_venta", StringType()),
    StructField("cantidad", IntegerType()),
    StructField("id_tienda", StringType()),
    StructField("fecha_hora", TimestampType())
])

schema_productos = StructType([
    StructField("id_producto", StringType()),
    StructField("nombre_producto", StringType()),
    StructField("sku", StringType()),
    StructField("categoria", StringType()),
    StructField("subcategoria", StringType()),
    StructField("talla", StringType()),
    StructField("color", StringType()),
    StructField("costo", DecimalType(18,2)),
    StructField("precio_venta", DecimalType(18,2))
])

schema_clientes = StructType([
    StructField("id_cliente", StringType()),
    StructField("nombre", StringType()),
    StructField("apellido", StringType()),
    StructField("email", StringType()),
    StructField("genero", StringType()),
    StructField("fecha_nacimiento", DateType()),
    StructField("localidad", StringType()),
    StructField("fecha_registro", DateType())
])

In [0]:

# ============================================================================
# 3. CAPA BRONCE: INGESTA CON AUTO LOADER
# ============================================================================
# Usamos cloudFiles para capturar datos nuevos automáticamente sin re-procesar

print("Iniciando Ingesta Bronce...")

# --- Ingesta de Clientes (CSV) ---
df_bronze_clientes = (spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("header", "true")
    .option("cloudFiles.schemaLocation", f"{SCHEMAS}/schema_bronze_clientes")
    .schema(schema_clientes)
    .load(f"{raw_path}/clientes")
    .withColumn("ingestion_time", current_timestamp())
    .withColumn("source_file", col("_metadata.file_name")))

query_bronze_clientes = (df_bronze_clientes.writeStream
    .format("delta")
    .option("checkpointLocation", f"{checkpoint_path}/bronze_clientes")
    .outputMode("append")
    .trigger(availableNow=True)
    .table(f"{CATALOG}.{BRONCE_SCHEMA}.bronze_clientes"))

# --- Ingesta de Productos (CSV) ---
df_bronze_productos = (spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("header", "true")
    .option("cloudFiles.schemaLocation", f"{SCHEMAS}/schema_bronze_productos")
    .schema(schema_productos)
    .load(f"{raw_path}/productos")
    .withColumn("ingestion_time", current_timestamp())
    .withColumn("source_file", col("_metadata.file_name")))

query_bronze_productos = (df_bronze_productos.writeStream
    .format("delta")
    .option("checkpointLocation", f"{checkpoint_path}/bronze_productos")
    .outputMode("append")
    .trigger(availableNow=True)
    .table(f"{CATALOG}.{BRONCE_SCHEMA}.bronze_productos"))

# --- Ingesta de Transacciones (JSON) ---
df_bronze_transacciones = (spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", f"{SCHEMAS}/schema_bronze_transacciones")
    .schema(schema_transacciones)
    .load(f"{raw_path}/transacciones")
    .withColumn("ingestion_time", current_timestamp())
    .withColumn("source_file", col("_metadata.file_name")))

query_bronze_transacciones = (df_bronze_transacciones.writeStream
    .format("delta")
    .option("checkpointLocation", f"{checkpoint_path}/bronze_transacciones")
    .outputMode("append")
    .trigger(availableNow=True)
    .table(f"{CATALOG}.{BRONCE_SCHEMA}.bronze_transacciones"))


query_bronze_productos.awaitTermination()
query_bronze_transacciones.awaitTermination()
query_bronze_clientes.awaitTermination()

In [0]:

# ============================================================================
# 4. CAPA SILVER: LIMPIEZA Y TRANSFORMACIÓN
# ============================================================================
# Leemos desde las tablas Delta Bronze de forma incremental

print("Iniciando Procesamiento Silver...")
# Silver Clientes: Tipado de datos
df_silver_clientes = (spark.readStream.table(f"{CATALOG}.{BRONCE_SCHEMA}.bronze_clientes")
    .dropDuplicates(["id_cliente"])
    .filter(col("id_cliente").isNotNull())
    .withColumn(
        "edad", 
        floor(datediff(to_date(lit(FECHA_CORTE)), col("fecha_nacimiento")) / 365.25)
    ).withColumn(
        "rango_edad",
        when(col("edad").isNull(), "N-A") # <--- Manejo de vacíos
        .when(col("edad") < 18, "Menor")
        .when((col("edad") >= 18) & (col("edad") <= 24), "18-24")
        .when((col("edad") >= 25) & (col("edad") <= 34), "25-34")
        .when((col("edad") >= 35) & (col("edad") <= 44), "35-44")
        .otherwise("+45")
    )
    .select(
        col("id_cliente"),
        col("nombre"),
        col("apellido"),
        col("localidad"),
        col("genero"),
        col("rango_edad"),
        to_date(col("fecha_registro")).alias("fecha_registro")
    ))

# MERGE idempotente por clave de negocio: si el registro ya existe, lo actualiza;
# si no, lo inserta. Permite reiniciar el stream desde el batch 0 sin duplicar filas
# (los checkpoints anteriores quedaron inservibles: estado sin offsets/commits).
def upsert_silver(batch_df, batch_id, target, key):
    if batch_df.isEmpty():
        return
    if spark.catalog.tableExists(target):
        (DeltaTable.forName(spark, target).alias("t")
            .merge(batch_df.alias("s"), f"t.{key} = s.{key}")
            .whenMatchedUpdateAll()
            .whenNotMatchedInsertAll()
            .execute())
    else:
        batch_df.write.format("delta").saveAsTable(target)

# Checkpoint nuevo: el anterior contiene estado de un run fallido y Spark
# rechaza arrancar en batch 0 con el state store no vacío.
query_silver_clientes = (df_silver_clientes.writeStream
    .foreachBatch(lambda df, i: upsert_silver(df, i, f"{CATALOG}.{SILVER_SCHEMA}.silver_clientes", "id_cliente"))
    .option("checkpointLocation", f"{checkpoint_path}/silver_clientes")
    .trigger(availableNow=True)
    .start())

# Silver Productos: Casting de precios y costos
df_silver_productos = (spark.readStream.table(f"{CATALOG}.{BRONCE_SCHEMA}.bronze_productos")
    .dropDuplicates(["id_producto"]) #Eliminando duplicados
    .filter(col("id_producto").isNotNull()) #Filtrando id no null
    .select(
        col("id_producto"),
        col("nombre_producto"),
        col("categoria"),
        col("costo").cast("decimal(18,2)"),
        col("precio_venta").cast("decimal(18,2)")
    ))

query_silver_productos = (df_silver_productos.writeStream
    .foreachBatch(lambda df, i: upsert_silver(df, i, f"{CATALOG}.{SILVER_SCHEMA}.silver_productos", "id_producto"))
    .option("checkpointLocation", f"{checkpoint_path}/silver_productos")
    .trigger(availableNow=True)
    .start())

# Silver Transacciones:
df_silver_transacciones = (spark.readStream.table(f"{CATALOG}.{BRONCE_SCHEMA}.bronze_transacciones")
    .dropDuplicates(["id_transaccion"]) #Eliminando duplicados
    .filter(col("id_transaccion").isNotNull()) #Filtrando id no nulls
    .select(
        col("id_transaccion"),
        col("id_cliente"),
        col("id_tienda"),
        col("id_producto"),
        col("estado_venta"),
        col("canal_venta"),
        col("cantidad").cast("int"),
        to_date(col("fecha_hora")).alias("fecha_venta"),
        col("metodo_pago")
    ))

query_silver_transacciones = (df_silver_transacciones.writeStream
    .foreachBatch(lambda df, i: upsert_silver(df, i, f"{CATALOG}.{SILVER_SCHEMA}.silver_transacciones", "id_transaccion"))
    .option("checkpointLocation", f"{checkpoint_path}/silver_transacciones")
    .trigger(availableNow=True)
    .start())

query_silver_clientes.awaitTermination()
query_silver_productos.awaitTermination()
query_silver_transacciones.awaitTermination()

In [0]:

# ============================================================================
# 5. CAPA GOLD: AGREGACIONES PARA NEGOCIO
# ============================================================================
# Unimos las tablas Silver
print("Iniciando Procesamiento Gold...")

df_s_trans = spark.readStream.table(f"{CATALOG}.{SILVER_SCHEMA}.silver_transacciones")
df_s_clie = spark.read.table(f"{CATALOG}.{SILVER_SCHEMA}.silver_clientes") # Tabla de referencia (estática)
df_s_prod = spark.read.table(f"{CATALOG}.{SILVER_SCHEMA}.silver_productos") # Tabla de referencia (estática)

#Gold reporte_ventas JOIN clientes - productos
gold_transacciones = (df_s_trans
    .join(df_s_clie, "id_cliente")
    .join(df_s_prod, "id_producto")
    .withColumn("ultima_actualizacion", current_timestamp())
    .withColumn("ingresos", (col("cantidad") * col("precio_venta")).cast("decimal(18,2)"))
    .withColumn("costos", (col("cantidad") * col("costo")).cast("decimal(18,2)"))
    .withColumn("ganancia", (col("ingresos") - col("costos")).cast("decimal(18,2)"))
    .select("id_transaccion","id_producto","id_cliente",
            "estado_venta","metodo_pago","canal_venta",
            "id_tienda","fecha_venta", col("costo").alias("costo_unitario"),
            col("precio_venta").alias("precio_unitario"),"cantidad","ingresos","costos","ganancia",
            "ultima_actualizacion")
    )
# MERGE idempotente: si la transacción ya existe, la actualiza; si no, la inserta.
# Permite re-ejecutar el pipeline sin duplicar filas (reintentos seguros).
TARGET_GOLD_VENTAS = f"{CATALOG}.{GOLD_SCHEMA}.gold_reporte_ventas"

def upsert_gold_reporte_ventas(batch_df, batch_id):
    if batch_df.isEmpty():
        return
    if spark.catalog.tableExists(TARGET_GOLD_VENTAS):
        (DeltaTable.forName(spark, TARGET_GOLD_VENTAS).alias("t")
            .merge(batch_df.alias("s"), "t.id_transaccion = s.id_transaccion")
            .whenMatchedUpdateAll()
            .whenNotMatchedInsertAll()
            .execute())
    else:
        batch_df.write.format("delta").saveAsTable(TARGET_GOLD_VENTAS)

# Checkpoint nuevo: el sink cambió de append a foreachBatch.
query_gold_1 = (gold_transacciones.writeStream
    .foreachBatch(upsert_gold_reporte_ventas)
    .option("checkpointLocation", f"{checkpoint_path}/gold_ventas_final_merge")
    .trigger(availableNow=True)
    .start())

# GOLD B : Ventas por Metodo de pago
gold_metodo_pago = (
    gold_transacciones
    .groupBy("metodo_pago")
    .agg(
        _sum("ingresos").alias("ingresos_totales"),
        _sum("ganancia").alias("ganancia_total"),
        approx_count_distinct("id_transaccion").alias("num_transacciones")
    )
    .orderBy(col("ingresos_totales").desc())
)


query_gold_2 = (gold_metodo_pago.writeStream
    .format("delta")
    .option("checkpointLocation", f"{checkpoint_path}/gold_ventas_por_metodo_pago")
    .outputMode("complete") # Reescribe la tabla completa
    .trigger(availableNow=True)
    .table(f"{CATALOG}.{GOLD_SCHEMA}.gold_ventas_por_metodo_pago")
)
# GOLD C : Ventas por CANAL
gold_canal_venta = (
    gold_transacciones
    .groupBy("canal_venta")
    .agg(
        _sum("ingresos").alias("ingresos_totales"),
        _sum("ganancia").alias("ganancia_total"),
        approx_count_distinct("id_transaccion").alias("num_transacciones")
    )
    .withColumn("ultima_actualizacion", current_timestamp())
    .orderBy(col("ingresos_totales").desc())
)

query_gold_3 = (gold_canal_venta.writeStream
    .format("delta")
    .option("checkpointLocation", f"{checkpoint_path}/gold_ventas_por_canal")
    .outputMode("complete")
    .trigger(availableNow=True)
    .table(f"{CATALOG}.{GOLD_SCHEMA}.gold_ventas_por_canal")
)

# GOLD D : Ventas Top Productos
gold_ventas_top_producto = (
    gold_transacciones
    .groupBy("id_producto")
    .agg(
        _sum("cantidad").alias("unidades_vendidas"),
        _sum("ingresos").alias("ingresos_totales"),
        _sum("ganancia").alias("ganancia_total"),
        approx_count_distinct("id_transaccion").alias("num_transacciones")
    )
    .withColumn("ultima_actualizacion", current_timestamp())
    .orderBy(col("ingresos_totales").desc()) 
    .limit(5)
)

query_gold_4 = (gold_ventas_top_producto.writeStream
    .format("delta")
    .option("checkpointLocation", f"{checkpoint_path}/gold_ventas_top_producto")
    .outputMode("complete")
    .trigger(availableNow=True)
    .table(f"{CATALOG}.{GOLD_SCHEMA}.gold_ventas_top_producto")
)

# GOLD D : Ventas por estado
gold_ventas_por_estado = (
    gold_transacciones
    .groupBy("estado_venta")
    .agg(
        _sum("cantidad").alias("unidades_vendidas"),
        _sum("ingresos").alias("ingresos_totales"),
        _sum("ganancia").alias("ganancia_total"),
        approx_count_distinct("id_transaccion").alias("num_transacciones")
    )
    .withColumn("ultima_actualizacion", current_timestamp())
    .orderBy(col("ingresos_totales").desc()) 
)

query_gold_5 = (gold_ventas_por_estado.writeStream
    .format("delta")
    .option("checkpointLocation", f"{checkpoint_path}/gold_ventas_por_estado")
    .outputMode("complete")
    .trigger(availableNow=True)
    .table(f"{CATALOG}.{GOLD_SCHEMA}.gold_ventas_por_estado")
)

# Esperar a que todas las queries de Gold terminen antes de cerrar la celda.
for q in [query_gold_1, query_gold_2, query_gold_3, query_gold_4, query_gold_5]:
    q.awaitTermination()

print("Procesamiento finalizado con éxito.")